# Build matrix B from EBR corepresentation decomposition table (HTML source).

In [29]:
import re
from pathlib import Path

import numpy as np

# Row order of B (1-based indices):
# Gamma_1..Gamma_8 (1-8), A_1..A_8 (9-16), M_1..M_8 (17-24),
# Z_1..Z_8 (25-32), R_1..R_4 (33-36), X_1..X_4 (37-40).
ROW_OFFSET = {"Gamma": 0, "A": 8, "M": 16, "Z": 24, "R": 32, "X": 36}
N_ROWS, N_COLS = 40, 36
KPOINT_TYPES = ("Gamma", "A", "M", "Z", "R", "X")


def parse_cell(cell_html: str) -> list[int]:
    """Extract representation sub-indices from one table cell; ignore (1) multiplicities."""
    parts = re.split(r"&nbsp;&oplus;&nbsp;|&oplus;", cell_html)
    indices = []
    for part in parts:
        match = re.search(r"<sub>(\d+)</sub>", part)
        if match:
            indices.append(int(match.group(1)))
    return indices


def _row_label(row_html: str) -> str:
    cells = re.findall(r"<td[^>]*>(.*?)</td>", row_html, re.DOTALL)
    if not cells:
        return ""
    return re.sub(r"<[^>]+>", "", cells[0]).strip()


def _identify_kpoint(label: str) -> str | None:
    label = label.replace("Γ", "Gamma").replace("&Gamma;", "Gamma")
    for rep in KPOINT_TYPES:
        if re.match(rf"^{rep}\s*:", label):
            return rep
    return None


def _extract_table_rows(html: str) -> list[str]:
    # BCS HTML may omit </tr>; split on <tr> instead of matching <tr>...</tr>.
    return [part for part in re.split(r"<tr>", html, flags=re.IGNORECASE) if "<td" in part]


def build_B_from_html(html: str) -> np.ndarray:
    data_rows: list[tuple[str, str]] = []
    for row_html in _extract_table_rows(html):
        rep_type = _identify_kpoint(_row_label(row_html))
        if rep_type is not None:
            data_rows.append((rep_type, row_html))

    if len(data_rows) != 6:
        labels = [_row_label(row) for _, row in data_rows]
        raise ValueError(f"Expected 6 k-point rows, found {len(data_rows)}: {labels}")

    first_cells = re.findall(r"<td[^>]*>(.*?)</td>", data_rows[0][1], re.DOTALL)
    n_label_cols = len(first_cells) - N_COLS
    if n_label_cols < 1:
        raise ValueError(
            f"Expected {N_COLS} data columns, but first k-point row has {len(first_cells)} cells"
        )

    B = np.zeros((N_ROWS, N_COLS), dtype=int)
    for col_idx in range(N_COLS):
        for rep_type, row_html in data_rows:
            cells = re.findall(r"<td[^>]*>(.*?)</td>", row_html, re.DOTALL)
            cell_html = cells[n_label_cols + col_idx]
            for sub_idx in parse_cell(cell_html):
                B[ROW_OFFSET[rep_type] + sub_idx - 1, col_idx] = 1
    return B


raw_path = Path("ebr_raw_data8133.txt")  # or "ebr_raw_data8133.txt"
html = raw_path.read_text(encoding="utf-8")
EBR = build_B_from_html(html)

print(f"source: {raw_path}")
print(f"EBR shape: {EBR.shape}")
print(f"Each indecomposable column has {EBR[:, 0].sum()} ones; decomposable columns have twice as many.")

source: ebr_raw_data8133.txt
EBR shape: (40, 36)
Each indecomposable column has 6 ones; decomposable columns have twice as many.


In [2]:
# Spot-check: column 0 is Gamma_4, A_4, M_4, R_2, X_2, Z_4 -> rows 4, 12, 20, 28, 34, 38 (1-based).
row_labels = (
    [f"Gamma_{i}" for i in range(1, 9)]
    + [f"A_{i}" for i in range(1, 9)]
    + [f"M_{i}" for i in range(1, 9)]
    + [f"Z_{i}" for i in range(1, 9)]
    + [f"R_{i}" for i in range(1, 5)]
    + [f"X_{i}" for i in range(1, 5)]
)

for i in range(N_COLS):
    active = np.where(EBR[:, i])[0]
    print(f"Column {i+1} active rows (1-based):", active + 1)
    print("Labels:", [row_labels[j] for j in active])

Column 1 active rows (1-based): [ 4 12 20 28 34 38]
Labels: ['Gamma_4', 'A_4', 'M_4', 'Z_4', 'R_2', 'X_2']
Column 2 active rows (1-based): [ 3 11 19 27 34 38]
Labels: ['Gamma_3', 'A_3', 'M_3', 'Z_3', 'R_2', 'X_2']
Column 3 active rows (1-based): [ 1  9 17 25 33 37]
Labels: ['Gamma_1', 'A_1', 'M_1', 'Z_1', 'R_1', 'X_1']
Column 4 active rows (1-based): [ 2 10 18 26 33 37]
Labels: ['Gamma_2', 'A_2', 'M_2', 'Z_2', 'R_1', 'X_1']
Column 5 active rows (1-based): [ 8 16 24 32 36 40]
Labels: ['Gamma_8', 'A_8', 'M_8', 'Z_8', 'R_4', 'X_4']
Column 6 active rows (1-based): [ 7 15 23 31 36 40]
Labels: ['Gamma_7', 'A_7', 'M_7', 'Z_7', 'R_4', 'X_4']
Column 7 active rows (1-based): [ 6 14 22 30 35 39]
Labels: ['Gamma_6', 'A_6', 'M_6', 'Z_6', 'R_3', 'X_3']
Column 8 active rows (1-based): [ 5 13 21 29 35 39]
Labels: ['Gamma_5', 'A_5', 'M_5', 'Z_5', 'R_3', 'X_3']
Column 9 active rows (1-based): [ 4 11 20 27 34 38]
Labels: ['Gamma_4', 'A_3', 'M_4', 'Z_3', 'R_2', 'X_2']
Column 10 active rows (1-based): [ 3 

In [20]:
# compatibility matrix CR s.t. CR B = 0
CR = np.zeros((16, N_ROWS), dtype=int)
CR[0, 0] = 1
CR[0, 1] = 1
CR[0, 24] = -1
CR[0, 25] = -1

CR[1, 2] = 1
CR[1, 3] = 1
CR[1, 26] = -1
CR[1, 27] = -1

CR[2, 4] = 1
CR[2, 5] = 1
CR[2, 28] = -1
CR[2, 29] = -1

CR[3, 6] = 1
CR[3, 7] = 1
CR[3, 30] = -1
CR[3, 31] = -1

CR[4, 8] = 1
CR[4, 9] = 1
CR[4, 16] = -1
CR[4, 17] = -1

CR[5, 10] = 1
CR[5, 11] = 1
CR[5, 18] = -1
CR[5, 19] = -1

CR[6, 12] = 1
CR[6, 13] = 1
CR[6, 20] = -1
CR[6, 21] = -1

CR[7, 14] = 1
CR[7, 15] = 1
CR[7, 22] = -1
CR[7, 23] = -1

CR[8, 32] = 1
CR[8, 36] = -1

CR[9, 33] = 1
CR[9, 37] = -1

CR[10, 34] = 1
CR[10, 38] = -1

CR[11, 35] = 1
CR[11, 39] = -1

CR[12, 0] = 1
CR[12, 1] = 1
CR[12, 2] = 1
CR[12, 3] = 1
CR[12, 32] = -1
CR[12, 33] = -1

CR[13, 4] = 1
CR[13, 5] = 1
CR[13, 6] = 1
CR[13, 7] = 1
CR[13, 34] = -1
CR[13, 35] = -1

CR[14, 8] = 1
CR[14, 9] = 1
CR[14, 10] = 1
CR[14, 11] = 1
CR[14, 32] = -1
CR[14, 33] = -1

CR[15, 12] = 1
CR[15, 13] = 1
CR[15, 14] = 1
CR[15, 15] = 1
CR[15, 34] = -1
CR[15, 35] = -1

## Smith normal form of `EBR`

For integer matrices, the Smith decomposition is
$$\mathrm{EBR}=L\,\Lambda\,R,$$
where $L\in\mathbb{Z}^{40\times 40}$ and $R\in\mathbb{Z}^{36\times 36}$ are **unimodular** (determinant $\pm1$), and $\Lambda\in\mathbb{Z}^{40\times 36}$ is diagonal with
$$\Lambda=\mathrm{diag}(\lambda_1,\ldots,\lambda_r,0,\ldots,0),\qquad 1\le\lambda_1\mid\cdots\mid\lambda_r.$$

**Important:** $L$ and $R$ are **not unique**, and a raw elimination output need not be the *canonical* Smith diagonal. What is unique (up to sign) are the **invariant factors** from `invariant_factors(EBR)`.

- A diagonal entry $\lambda$ and $-\lambda$ give the same factor $\mathbb{Z}_{|\lambda|}$ (multiply that row by $-1$).
- `smith_decomposition` returns the **canonical** Smith diagonal $\Lambda$ with non-negative entries and $\lambda_i\mid\lambda_{i+1}$, together with unimodular $L,R$ such that $\mathrm{EBR}=L\Lambda R$.

- **Rank**: $r$ equals the number of nonzero invariant factors, and also `EBR.rank()` over $\mathbb{Q}$.
- **Symmetry-indicator group**: from invariant factors with $\lambda_i>1$.
- **Indicators**: $z_i(B)=[L^{-1}B]_i\bmod \lambda_i$ using `L_EBR` and the diagonal of `Lambda_EBR` from the same `smith_decomposition` call.

In [21]:
# check compatibility relationship
CR @ EBR

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],


In [22]:
import sympy as sp
from sympy.matrices.normalforms import invariant_factors, smith_normal_form
from sympy.polys.matrices import DomainMatrix
from sympy.polys.matrices.normalforms import smith_normal_decomp as _smith_normal_decomp


def smith_normal_decomp(m, domain=None):
    dM = DomainMatrix.from_Matrix(m)
    if domain is not None:
        dM = dM.convert_to(domain)
    a, s, t = _smith_normal_decomp(dM)
    return a.to_Matrix(), s.to_Matrix(), t.to_Matrix()


def smith_decomposition(A: np.ndarray):
    """Return unimodular L, R and canonical SNF Lambda with A = L @ Lambda @ R.

    Lambda is diagonal with non-negative entries lambda_1 | ... | lambda_r
    on the leading rank positions, then zeros.
    """
    M = sp.Matrix(np.asarray(A, dtype=int).tolist())
    Lambda, S, T = smith_normal_decomp(M, domain=sp.ZZ)
    L, R = S.inv(), T.inv()
    assert (M - L * Lambda * R).is_zero_matrix
    assert Lambda == smith_normal_form(M)

    r = min(Lambda.shape)
    prev = 1
    for i in range(r):
        d = int(Lambda[i, i])
        if d < 0:
            raise ValueError(f"SNF diagonal entry Lambda[{i},{i}] = {d} is negative")
        if d == 0:
            break
        if i > 0 and d % prev != 0:
            raise ValueError(f"SNF divisibility failed: {prev} does not divide {d}")
        prev = d
    return L, Lambda, R


M_EBR = sp.Matrix(EBR.tolist())

# Rank over Q (equals the number of nonzero Smith invariants).
rank_EBR = M_EBR.rank()

# Canonical invariant factors: lambda_1 | ... | lambda_r.
invariant_factors_EBR = tuple(int(x) for x in invariant_factors(M_EBR) if int(x) != 0)
rank_EBR_snf = len(invariant_factors_EBR)

L_EBR, Lambda_EBR, R_EBR = smith_decomposition(EBR)
assert (L_EBR * Lambda_EBR * R_EBR - M_EBR).is_zero_matrix
assert list(Lambda_EBR.diagonal()[:rank_EBR_snf]) == list(invariant_factors_EBR)

print(f"rank(EBR) = {rank_EBR}")
print(f"invariant factors (canonical) = {invariant_factors_EBR}")
print(f"Lambda_EBR diagonal = {Lambda_EBR.diagonal()}")
print(f"det(L_EBR) = {int(L_EBR.det())}, det(R_EBR) = {int(R_EBR.det())}")
print("nontrivial SI moduli:", [lam for lam in invariant_factors_EBR if lam > 1])
nontrivial_si_rows = [
    (i + 1, int(Lambda_EBR[i, i]))
    for i in range(min(Lambda_EBR.shape))
    if int(Lambda_EBR[i, i]) > 1
]
print("nontrivial SI rows (L_EBR / Lambda_EBR order):", nontrivial_si_rows)

rank(EBR) = 24
invariant factors (canonical) = (1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 4, 4)
Lambda_work diagonal (from elimination) = Matrix([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, -1, 1, 0, 1, 1, 4, 0, 1, 1, 4, 0, 2, 2, 0, 0, 2, -2, 0, 0, 0, 0, 0, 0]])
Lambda_EBR diagonal (canonical SNF)    = Matrix([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 4, -4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
det(L_EBR) = -1, det(R_EBR) = 1
nontrivial SI moduli (canonical): [2, 2, 2, 2, 4, 4]
nontrivial SI rows (L_EBR / Lambda_work order): [(19, 4), (23, 4), (25, 2), (26, 2), (29, 2), (30, 2)]


In [23]:
# Row moduli paired with L_EBR (same order as the canonical SNF diagonal).
si_moduli_EBR = tuple(
    int(Lambda_EBR[i, i]) for i in range(min(Lambda_EBR.shape))
)


def symmetry_indicators(B: np.ndarray):
    """Return SI coordinates z_i(B) = [L_EBR^{-1} B]_i mod lambda_i for lambda_i > 1.

    L_EBR and si_moduli_EBR come from the same smith_decomposition call;
    Lambda_EBR is the canonical Smith normal form.
    """
    B = np.asarray(B, dtype=int).reshape(-1)
    y = (L_EBR.inv() * sp.Matrix(B.tolist())).applyfunc(int)
    indicators = {}
    for i, lam in enumerate(si_moduli_EBR, start=1):
        if lam > 1:
            indicators[i] = int(y[i - 1]) % lam
    return indicators


# Example: any EBR column is an atomic band representation, so all indicators vanish.
for col in range(36):
    z = symmetry_indicators(EBR[:, col])
    print(f"column {col}: indicators = {z}")

# Inspect the diagonal Smith matrix (first 24 rows/cols are the active block).
Lambda_EBR[:rank_EBR, :rank_EBR]

column 0: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 1: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 2: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 3: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 4: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 5: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 6: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 7: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 8: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 9: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 10: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 11: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 12: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 13: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
column 14: indicators = {19: 0, 23: 0, 25: 0, 26: 0, 29: 0, 30: 0}
colum

Matrix([
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0,

Let's then test whether the ith column could correctly give SIs.
Consider a two-band system with $S_4$ eigenvalues:

$\Gamma$: $e^{\pm i\frac{3\pi}{4}}$

$Z$: $e^{\pm i \frac{\pi}{4}}$

$M$: $e^{\pm i\frac{3\pi}{4}}$

$A$: $e^{\pm i\frac{3\pi}{4}}$

$R$: Here it's $(S_4)^2$ eigenvalues $e^{\pm i\frac{\pi}{2}}$

$X$: $(S_4)^2$ eigenvalues $e^{\pm i\frac{\pi}{2}}$

We should first write down the corresponding symmetry data (Note: the symmetry data is composed of the multiplicities of irreducible small group correps $\Gamma_1$, $\Gamma_2$, $\Gamma_3$, $\Gamma_4$, $\bar{\Gamma}_5$, $\bar{\Gamma}_6$, $\bar{\Gamma}_7$, $\bar{\Gamma}_8$,similar for $A$, $M$, $Z$, and $R_1$, $R_2$, $\bar{R}_3$, $\bar{R}_4$, $X_1$, $X_2$, $\bar{X}_3$, $\bar{X}_4$). The $\bar{\Gamma}_5$ band has $S_4$-eigenvalue $e^{i\frac{3\pi}{4}}$, $\bar{\Gamma}_6$ band has $S_4$-eigenvalue $e^{-i\frac{\pi}{4}}$, $\bar{\Gamma}_7$ band has $S_4$-eigenvalue $e^{-i\frac{3\pi}{4}}$, $\bar{\Gamma}_8$ band has $S_4$-eigenvalue $e^{i\frac{\pi}{4}}$. $\bar{R}_3$ band has $(S_4)^2$-eigenvalue $e^{-i\frac{\pi}{2}}$, $\bar{R}_4$ band has $(S_4)^2$-eigenvalue $e^{i\frac{\pi}{2}}$. 

In [25]:
B_topo = np.zeros((40,1), dtype=int)
B_topo[4]=1
B_topo[6]=1
B_topo[12]=1
B_topo[14]=1
B_topo[20]=1
B_topo[22]=1
B_topo[29]=1
B_topo[31]=1
B_topo[34]=1
B_topo[35]=1
B_topo[38]=1
B_topo[39]=1
z = symmetry_indicators(B_topo)
print(CR @ B_topo)
print(z)

[[0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]]
{19: 0, 23: 0, 25: 0, 26: 0, 29: 1, 30: 1}


In [26]:
def integer_kernel_basis(C: np.ndarray) -> list[np.ndarray]:
    """Return an integer basis for ker(C), i.e. C @ v = 0."""
    null_vecs = sp.Matrix(C.tolist()).nullspace()
    basis: list[np.ndarray] = []
    for v in null_vecs:
        denoms = [x.q for x in v if x.q != 1]
        scale = int(sp.ilcm(*denoms)) if denoms else 1
        basis.append(np.array([int(scale * x) for x in v], dtype=int))
    return basis


def nonnegative_compatible_generators(
    rng: np.random.Generator,
    *,
    kernel_basis: list[np.ndarray] | None = None,
    max_kernel_coeff: int = 6,
    n_kernel_trials: int = 5000,
) -> list[np.ndarray]:
    """Non-negative integer generators for compatible symmetry data (CR @ B = 0, B >= 0)."""
    generators: list[np.ndarray] = [EBR[:, j].copy() for j in range(N_COLS)]
    generators.append(B_topo.reshape(-1).copy())

    if kernel_basis is None:
        kernel_basis = integer_kernel_basis(CR)

    seen = {tuple(g.tolist()) for g in generators}
    for _ in range(n_kernel_trials):
        coeffs = rng.integers(0, max_kernel_coeff + 1, size=len(kernel_basis))
        if not np.any(coeffs):
            continue
        vec = sum(int(c) * b for c, b in zip(coeffs, kernel_basis))
        if np.all(vec >= 0) and np.any(vec > 0):
            key = tuple(vec.tolist())
            if key not in seen:
                seen.add(key)
                generators.append(vec)
    return generators


def random_compatible_B(
    rng: np.random.Generator,
    *,
    coeff_range: int = 3,
    max_terms: int = 8,
    generators: list[np.ndarray] | None = None,
) -> np.ndarray:
    """Random non-negative integer B in ker(CR), shape (40, 1)."""
    if generators is None:
        generators = NONNEG_GENERATORS
    n_pick = min(max_terms, len(generators))
    idx = rng.choice(len(generators), size=n_pick, replace=False)
    coeffs = rng.integers(1, coeff_range + 1, size=n_pick)
    B = sum(int(c) * generators[i] for c, i in zip(coeffs, idx))
    return B.reshape(-1, 1)


def active_rep_labels(B: np.ndarray) -> list[str]:
    B = np.asarray(B, dtype=int).reshape(-1)
    return [f"{row_labels[i]}={B[i]}" for i in np.nonzero(B)[0]]


RNG = np.random.default_rng(42)
KER_CR = integer_kernel_basis(CR)
NONNEG_GENERATORS = nonnegative_compatible_generators(RNG, kernel_basis=KER_CR)
print(f"ker(CR) dimension = {len(KER_CR)}")
print(f"non-negative compatible generators = {len(NONNEG_GENERATORS)}")

N_SAMPLES = 8
print(
    f"\nRandom compatible symmetry data ({N_SAMPLES} samples, "
    f"non-negative integer entries):\n"
)
for k in range(N_SAMPLES):
    B_rand = random_compatible_B(RNG, generators=NONNEG_GENERATORS)
    assert np.all(B_rand >= 0), "entries must be non-negative"
    assert np.all(CR @ B_rand == 0), "compatibility check failed"
    z = symmetry_indicators(B_rand)
    print(f"--- sample {k + 1} ---")
    print("nonzero entries:", active_rep_labels(B_rand))
    print("symmetry indicators:", z)
    print()

print("--- B_topo (reference) ---")
print("nonzero entries:", active_rep_labels(B_topo))
print("symmetry indicators:", symmetry_indicators(B_topo))

ker(CR) dimension = 24
non-negative compatible generators = 44

Random compatible symmetry data (8 samples, non-negative integer entries):

--- sample 1 ---
nonzero entries: ['Gamma_1=17', 'Gamma_2=1', 'Gamma_3=12', 'Gamma_4=11', 'Gamma_5=22', 'Gamma_6=15', 'Gamma_7=9', 'Gamma_8=7', 'A_1=9', 'A_2=20', 'A_3=2', 'A_4=10', 'A_5=16', 'A_7=18', 'A_8=19', 'M_1=24', 'M_2=5', 'M_3=3', 'M_4=9', 'M_5=11', 'M_6=5', 'M_7=13', 'M_8=24', 'Z_1=8', 'Z_2=10', 'Z_3=6', 'Z_4=17', 'Z_5=13', 'Z_6=24', 'Z_7=6', 'Z_8=10', 'R_1=24', 'R_2=17', 'R_3=29', 'R_4=24', 'X_1=24', 'X_2=17', 'X_3=29', 'X_4=24']
symmetry indicators: {19: 1, 23: 1, 25: 1, 26: 0, 29: 0, 30: 0}

--- sample 2 ---
nonzero entries: ['Gamma_1=2', 'Gamma_3=6', 'Gamma_4=5', 'Gamma_6=3', 'Gamma_7=3', 'Gamma_8=2', 'A_1=2', 'A_2=4', 'A_3=2', 'A_4=5', 'A_5=3', 'A_7=5', 'M_1=4', 'M_2=2', 'M_4=7', 'M_5=3', 'M_7=2', 'M_8=3', 'Z_2=2', 'Z_3=4', 'Z_4=7', 'Z_6=3', 'Z_8=5', 'R_1=8', 'R_2=5', 'R_3=5', 'R_4=3', 'X_1=8', 'X_2=5', 'X_3=5', 'X_4=3']
symmetry ind

# Double SI 
Below, in accordance with the article "Magnetic Topological Quantum Chemistry", I'm going to solely consider the double SI groups. Then the Smith Normal Form of EBR is expected to give double SI group $Z_4\times Z_2^2$ rather than abovementioned $Z_4^2\times Z_2^4$. 

The components of $B$ are reduced to:
Row order of B (1-based indices):
$\bar{\Gamma}_5$...$\bar{\Gamma}_8$ (1-4), $\bar{A}_5$...$\bar{A}_8$ (5-8), $\bar{M}_5$...$\bar{M}_8$ (9-12),
$\bar{Z}_5$...$\bar{Z}_8$ (13-16), $\bar{R}_3$,$\bar{R}_4$ (17-18), $\bar{X}_3$, $\bar{X}_4$ (19-20).

EBR is then (20, 36); CR is then (8, 20).

**To reproduce the full analysis on the reduced basis, run only the last two code cells.**


In [2]:
# Self-contained setup for double-SI analysis (run this cell, then the next).
import re
from pathlib import Path

import numpy as np
import sympy as sp
from sympy.matrices.normalforms import invariant_factors, smith_normal_form
from sympy.polys.matrices import DomainMatrix
from sympy.polys.matrices.normalforms import smith_normal_decomp as _smith_normal_decomp


def smith_normal_decomp(m, domain=None):
    """Return (Lambda, S, T) with S @ M @ T = Lambda (Smith normal form)."""
    dM = DomainMatrix.from_Matrix(m)
    if domain is not None:
        dM = dM.convert_to(domain)
    a, s, t = _smith_normal_decomp(dM)
    return a.to_Matrix(), s.to_Matrix(), t.to_Matrix()


FULL_N_ROWS = 40
N_COLS = 36
ROW_OFFSET = {"Gamma": 0, "A": 8, "M": 16, "Z": 24, "R": 32, "X": 36}
KPOINT_TYPES = ("Gamma", "A", "M", "Z", "R", "X")


def parse_cell(cell_html: str) -> list[int]:
    parts = re.split(r"&nbsp;&oplus;&nbsp;|&oplus;", cell_html)
    indices = []
    for part in parts:
        match = re.search(r"<sub>(\d+)</sub>", part)
        if match:
            indices.append(int(match.group(1)))
    return indices


def _row_label(row_html: str) -> str:
    cells = re.findall(r"<td[^>]*>(.*?)</td>", row_html, re.DOTALL)
    if not cells:
        return ""
    return re.sub(r"<[^>]+>", "", cells[0]).strip()


def _identify_kpoint(label: str) -> str | None:
    label = label.replace("Γ", "Gamma").replace("&Gamma;", "Gamma")
    for rep in KPOINT_TYPES:
        if re.match(rf"^{rep}\s*:", label):
            return rep
    return None


def _extract_table_rows(html: str) -> list[str]:
    return [part for part in re.split(r"<tr>", html, flags=re.IGNORECASE) if "<td" in part]


def build_B_from_html(html: str) -> np.ndarray:
    data_rows: list[tuple[str, str]] = []
    for row_html in _extract_table_rows(html):
        rep_type = _identify_kpoint(_row_label(row_html))
        if rep_type is not None:
            data_rows.append((rep_type, row_html))

    if len(data_rows) != 6:
        labels = [_row_label(row) for _, row in data_rows]
        raise ValueError(f"Expected 6 k-point rows, found {len(data_rows)}: {labels}")

    first_cells = re.findall(r"<td[^>]*>(.*?)</td>", data_rows[0][1], re.DOTALL)
    n_label_cols = len(first_cells) - N_COLS
    if n_label_cols < 1:
        raise ValueError(
            f"Expected {N_COLS} data columns, but first k-point row has {len(first_cells)} cells"
        )

    B = np.zeros((FULL_N_ROWS, N_COLS), dtype=int)
    for col_idx in range(N_COLS):
        for rep_type, row_html in data_rows:
            cells = re.findall(r"<td[^>]*>(.*?)</td>", row_html, re.DOTALL)
            cell_html = cells[n_label_cols + col_idx]
            for sub_idx in parse_cell(cell_html):
                B[ROW_OFFSET[rep_type] + sub_idx - 1, col_idx] = 1
    return B


def smith_decomposition(A: np.ndarray):
    """Return unimodular L, R and canonical SNF Lambda with A = L @ Lambda @ R.

    Lambda is diagonal with non-negative entries lambda_1 | ... | lambda_r
    on the leading rank positions, then zeros.
    """
    M = sp.Matrix(np.asarray(A, dtype=int).tolist())
    Lambda, S, T = smith_normal_decomp(M, domain=sp.ZZ)
    L, R = S.inv(), T.inv()
    assert (M - L * Lambda * R).is_zero_matrix
    assert Lambda == smith_normal_form(M)

    r = min(Lambda.shape)
    prev = 1
    for i in range(r):
        d = int(Lambda[i, i])
        if d < 0:
            raise ValueError(f"SNF diagonal entry Lambda[{i},{i}] = {d} is negative")
        if d == 0:
            break
        if i > 0 and d % prev != 0:
            raise ValueError(f"SNF divisibility failed: {prev} does not divide {d}")
        prev = d
    return L, Lambda, R


def integer_kernel_basis(C: np.ndarray) -> list[np.ndarray]:
    null_vecs = sp.Matrix(C.tolist()).nullspace()
    basis: list[np.ndarray] = []
    for v in null_vecs:
        denoms = [x.q for x in v if x.q != 1]
        scale = int(sp.ilcm(*denoms)) if denoms else 1
        basis.append(np.array([int(scale * x) for x in v], dtype=int))
    return basis


def nonnegative_compatible_generators(
    rng: np.random.Generator,
    *,
    kernel_basis: list[np.ndarray] | None = None,
    max_kernel_coeff: int = 6,
    n_kernel_trials: int = 5000,
) -> list[np.ndarray]:
    generators: list[np.ndarray] = [
        EBR[:, j].copy() for j in range(N_COLS) if np.any(EBR[:, j])
    ]
    generators.append(B_topo.reshape(-1).copy())

    if kernel_basis is None:
        kernel_basis = integer_kernel_basis(CR)

    seen = {tuple(g.tolist()) for g in generators}
    for _ in range(n_kernel_trials):
        coeffs = rng.integers(0, max_kernel_coeff + 1, size=len(kernel_basis))
        if not np.any(coeffs):
            continue
        vec = sum(int(c) * b for c, b in zip(coeffs, kernel_basis))
        if np.all(vec >= 0) and np.any(vec > 0):
            key = tuple(vec.tolist())
            if key not in seen:
                seen.add(key)
                generators.append(vec)
    return generators


def random_compatible_B(
    rng: np.random.Generator,
    *,
    coeff_range: int = 3,
    max_terms: int = 8,
    generators: list[np.ndarray] | None = None,
) -> np.ndarray:
    if generators is None:
        generators = NONNEG_GENERATORS
    n_pick = min(max_terms, len(generators))
    idx = rng.choice(len(generators), size=n_pick, replace=False)
    coeffs = rng.integers(1, coeff_range + 1, size=n_pick)
    B = sum(int(c) * generators[i] for c, i in zip(coeffs, idx))
    return B.reshape(-1, 1)


def active_rep_labels(B: np.ndarray) -> list[str]:
    B = np.asarray(B, dtype=int).reshape(-1)
    return [f"{row_labels[i]}={B[i]}" for i in np.nonzero(B)[0]]


# --- load full EBR, then reduce to double-SI row basis ---
raw_path = Path("ebr_raw_data8133.txt")
html = raw_path.read_text(encoding="utf-8")
EBR = build_B_from_html(html)

OLD_ROWS = (
    list(range(4, 8))
    + list(range(12, 16))
    + list(range(20, 24))
    + list(range(28, 32))
    + [34, 35]
    + [38, 39]
)
N_ROWS = 20
EBR = EBR[OLD_ROWS, :].copy()

CR = np.zeros((8, N_ROWS), dtype=int)
CR[0, 0] = CR[0, 1] = 1
CR[0, 12] = CR[0, 13] = -1
CR[1, 2] = CR[1, 3] = 1
CR[1, 14] = CR[1, 15] = -1
CR[2, 4] = CR[2, 5] = 1
CR[2, 8] = CR[2, 9] = -1
CR[3, 6] = CR[3, 7] = 1
CR[3, 10] = CR[3, 11] = -1
CR[4, 16] = 1
CR[4, 18] = -1
CR[5, 17] = 1
CR[5, 19] = -1
CR[6, 0:4] = 1
CR[6, 16] = CR[6, 17] = -1
CR[7, 4:8] = 1
CR[7, 16] = CR[7, 17] = -1

row_labels = (
    [f"Gamma_{i}" for i in range(5, 9)]
    + [f"A_{i}" for i in range(5, 9)]
    + [f"M_{i}" for i in range(5, 9)]
    + [f"Z_{i}" for i in range(5, 9)]
    + [f"R_{i}" for i in range(3, 5)]
    + [f"X_{i}" for i in range(3, 5)]
)

# Topological test band (cell 9), projected onto the reduced basis.
B_topo = np.zeros((N_ROWS, 1), dtype=int)
for idx in (0, 2, 4, 6, 8, 10, 13, 15, 16, 17, 18, 19):
    B_topo[idx] = 1

print(f"source: {raw_path}")
print(f"CR shape: {CR.shape}, EBR shape: {EBR.shape}")
print(f"CR @ EBR all zero: {np.all(CR @ EBR == 0)}")
print(
    f"Indecomposable example (col 5): {EBR[:, 4].sum()} ones; "
    f"decomposable (col 35): {EBR[:, 34].sum()} ones."
)

print("\nActive EBR columns in the reduced basis:")
for col in range(N_COLS):
    if not np.any(EBR[:, col]):
        continue
    active = np.where(EBR[:, col])[0]
    print(f"Column {col + 1} rows (1-based):", active + 1)
    print("Labels:", [row_labels[j] for j in active])


source: ebr_raw_data8133.txt
CR shape: (8, 20), EBR shape: (20, 36)
CR @ EBR all zero: True
Indecomposable example (col 5): 6 ones; decomposable (col 35): 12 ones.

Active EBR columns in the reduced basis:
Column 5 rows (1-based): [ 4  8 12 16 18 20]
Labels: ['Gamma_8', 'A_8', 'M_8', 'Z_8', 'R_4', 'X_4']
Column 6 rows (1-based): [ 3  7 11 15 18 20]
Labels: ['Gamma_7', 'A_7', 'M_7', 'Z_7', 'R_4', 'X_4']
Column 7 rows (1-based): [ 2  6 10 14 17 19]
Labels: ['Gamma_6', 'A_6', 'M_6', 'Z_6', 'R_3', 'X_3']
Column 8 rows (1-based): [ 1  5  9 13 17 19]
Labels: ['Gamma_5', 'A_5', 'M_5', 'Z_5', 'R_3', 'X_3']
Column 13 rows (1-based): [ 4  7 12 15 18 20]
Labels: ['Gamma_8', 'A_7', 'M_8', 'Z_7', 'R_4', 'X_4']
Column 14 rows (1-based): [ 3  8 11 16 18 20]
Labels: ['Gamma_7', 'A_8', 'M_7', 'Z_8', 'R_4', 'X_4']
Column 15 rows (1-based): [ 2  5 10 13 17 19]
Labels: ['Gamma_6', 'A_5', 'M_6', 'Z_5', 'R_3', 'X_3']
Column 16 rows (1-based): [ 1  6  9 14 17 19]
Labels: ['Gamma_5', 'A_6', 'M_5', 'Z_6', 'R_3

In [7]:
# Full double-SI analysis (run after the previous cell only).

# --- compatibility check (cell 5) ---
assert np.all(CR @ EBR == 0)
print("CR @ EBR = 0:", True)

# --- Smith normal form (cell 6) ---
M_EBR = sp.Matrix(EBR.tolist())
rank_EBR = M_EBR.rank()
invariant_factors_EBR = tuple(int(x) for x in invariant_factors(M_EBR) if int(x) != 0)
rank_EBR_snf = len(invariant_factors_EBR)

L_EBR, Lambda_EBR, R_EBR = smith_decomposition(EBR)
assert (L_EBR * Lambda_EBR * R_EBR - M_EBR).is_zero_matrix
assert list(Lambda_EBR.diagonal()[:rank_EBR_snf]) == list(invariant_factors_EBR)

si_moduli_EBR = tuple(
    int(Lambda_EBR[i, i]) for i in range(min(Lambda_EBR.shape))
)


def symmetry_indicators(B: np.ndarray) -> dict[int, int]:
    """SI coordinates z_i(B) = [L_EBR^{-1} B]_i mod lambda_i for lambda_i > 1."""
    B = np.asarray(B, dtype=int).reshape(-1)
    y = (L_EBR.inv() * sp.Matrix(B.tolist())).applyfunc(int)
    indicators = {}
    for i, lam in enumerate(si_moduli_EBR, start=1):
        if lam > 1:
            indicators[i] = int(y[i - 1]) % lam
    return indicators


print(f"\nrank(EBR) = {rank_EBR}")
print(f"invariant factors (canonical) = {invariant_factors_EBR}")
print(f"Lambda_EBR diagonal = {Lambda_EBR.diagonal()}")
print(f"det(L_EBR) = {int(L_EBR.det())}, det(R_EBR) = {int(R_EBR.det())}")
print("nontrivial SI moduli:", [lam for lam in invariant_factors_EBR if lam > 1])
nontrivial_si_rows = [
    (i + 1, int(Lambda_EBR[i, i]))
    for i in range(min(Lambda_EBR.shape))
    if int(Lambda_EBR[i, i]) > 1
]
print("nontrivial SI rows (L_EBR / Lambda_EBR order):", nontrivial_si_rows)

# --- EBR column indicators (cell 7) ---
print("\nEBR column symmetry indicators (nonzero columns only):")
for col in range(N_COLS):
    if not np.any(EBR[:, col]):
        continue
    z = symmetry_indicators(EBR[:, col])
    print(f"column {col}: indicators = {z}")

# --- B_topo test (cell 9) ---
print("\nB_topo compatibility:", np.all(CR @ B_topo == 0))
print("B_topo symmetry indicators:", symmetry_indicators(B_topo))

# --- random compatible symmetry data (cell 10) ---
RNG = np.random.default_rng(42)
KER_CR = integer_kernel_basis(CR)
NONNEG_GENERATORS = nonnegative_compatible_generators(RNG, kernel_basis=KER_CR)
print(f"\nker(CR) dimension = {len(KER_CR)}")
print(f"non-negative compatible generators = {len(NONNEG_GENERATORS)}")

N_SAMPLES = 8
print(
    f"\nRandom compatible symmetry data ({N_SAMPLES} samples, "
    f"non-negative integer entries):\n"
)
for k in range(N_SAMPLES):
    B_rand = random_compatible_B(RNG, generators=NONNEG_GENERATORS)
    assert np.all(B_rand >= 0), "entries must be non-negative"
    assert np.all(CR @ B_rand == 0), "compatibility check failed"
    z = symmetry_indicators(B_rand)
    print(f"--- sample {k + 1} ---")
    print("nonzero entries:", active_rep_labels(B_rand))
    print("symmetry indicators:", z)
    print()

print("--- B_topo (reference) ---")
print("nonzero entries:", active_rep_labels(B_topo))
print("symmetry indicators:", symmetry_indicators(B_topo))

# Inspect the diagonal Smith matrix (active block).
Lambda_EBR[:rank_EBR, :rank_EBR]

CR @ EBR = 0: True

rank(EBR) = 12
invariant factors (canonical) = (1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 4)
Lambda_EBR diagonal = Matrix([[1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 4, 0, 0, 0, 0, 0, 0, 0, 0]])
det(L_EBR) = -1, det(R_EBR) = -1
nontrivial SI moduli: [2, 2, 4]
nontrivial SI rows (L_EBR / Lambda_EBR order): [(10, 2), (11, 2), (12, 4)]

EBR column symmetry indicators (nonzero columns only):
column 4: indicators = {10: 0, 11: 0, 12: 0}
column 5: indicators = {10: 0, 11: 0, 12: 0}
column 6: indicators = {10: 0, 11: 0, 12: 0}
column 7: indicators = {10: 0, 11: 0, 12: 0}
column 12: indicators = {10: 0, 11: 0, 12: 0}
column 13: indicators = {10: 0, 11: 0, 12: 0}
column 14: indicators = {10: 0, 11: 0, 12: 0}
column 15: indicators = {10: 0, 11: 0, 12: 0}
column 20: indicators = {10: 0, 11: 0, 12: 0}
column 21: indicators = {10: 0, 11: 0, 12: 0}
column 22: indicators = {10: 0, 11: 0, 12: 0}
column 23: indicators = {10: 0, 11: 0, 12: 0}
column 28: indicators = {10: 0, 11: 0, 12: 0}
column 29: in

Matrix([
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4]])

As we can see, the `B_topo` gives 1,1,0 indicator, which is not what we expected $(z_{4S},\delta_{2S},z_2)=(0,0,1)$. This is because there could be many different $L$ to choose. And we aren't using the physical basis. Therefore, it's safer to calculate the numbers of eigenvalues of bands at high-symmetry momenta, based on the multiplicities read from symmetry data, to compute the SIs.

# Prove the completeness of $B_1$ and $B_2$ over arbitrary $B$

Consider the monoid $\mathcal{M}$ (allowed symmetry data, so an element $B$ is a positive integer vector) satisfying $CR\cdot B=0$ and $z_{4S}=0,\delta_{2S}=0$ (the expression of these two indicators can be found in Supplemental Information of Magnetic Topological Quantum Chemistry, obtained by eigenvalues of bands at high-symmetry momenta). 

I want to claim that B, added with some symmetry data of EBR, can always be spanned by $B_1$ and $B_2$, which are symmetry data satisfying $CR\cdot B_i=0$, $z_{4S}=0$ and $\delta_{2S}=0$, but with one-band and two bands respectively. (Here $B_1$ may not be the same as EBR, since the vanishing SIs isn't equivalent to trivial. Fragile topology and other stuffs may happen).

## Description
Consider the system with $S_4$ symmetry. Its symmetry indicator group is $Z_4\times Z_2\times Z_2$, denoted as $z_{4S},\delta_{2S},z_2$, whose values are determined from the number of bands with different $S_4$ eigenvalues at $\Gamma,Z,M,A$ and $(S_4)^2$ eigenvalues at $R$. 

Let $B$ be all the possible symmetry data of a system satisfying compatibility relationship $CR\cdot B=0$ and $z_{4S}=\delta_{2S}=0$. Let $B_1$ be all the possible symmetry data of a single band, satisfying $CR\cdot B_1=0$ and $z_{4S}=\delta_{2S}=z_2=0$ (the last equation should be automatically satisfied). Let $B_2$ be all the possible symmetry data of a two-band system, satisfying $CR\cdot B_2=0$ and $z_{4S}=\delta_{2S}=0$. I'm going to show that $B$, maybe added with an integer combination of EBRs, can always be written as an integer combination of $B_1$ and $B_2$. 

To be specific, I'm going to first solve the "basis" of the above $B$: I'll construct the $CR$ matrix. Then get the EBR matrix. Finally, I'll construct the two row vectors $P_1$ and $P_2$, derived from the SI formula in the article, imposing constraints $P_1B-4u=0$ and $P_2B-2v=0$ where $u$ and $v$ are integers. With the matrix $\begin{bmatrix}CR&0&0\\P_1&-4&0\\P_2&0&-2\end{bmatrix}$ that gives 0 when acting on $(B;u;v)$, I could obtain the Hilbert-basis: $H=(h_1,h_2,\cdots h_s)$. 

Next, I enumerate $B_1$ and $B_2$, whose band numbers are constrained via $\sum_a(\dim \sigma_a)B_a=\text{band num}$, also satisfying the constraints of $CR$ and SIs. Concatenate $B_1$ and $B_2$ to a matrix $G$

Finally, I'll show that there exist nonnegative integer matrices U,V such that: $GU=H+EV$

## Find the hilbert-basis for monoid of $B$

In [ ]:
# I have difficulties installing PyNormaliz, so this part is conducted in Ubuntu. And the output H is stored in ./hilbert_basis.npy
# H(2466, 20) is the output.

# import numpy as np
# from PyNormaliz import Cone

# # for z_{4S} indicator
# P_1 = np.array([0,0,0,0,1.5,-0.5,-1.5,0.5,0,0,0,0,1.5,-0.5,-1.5,0.5,-1.0,1.0,0,0])
# P_2 = np.array([-1,0,1,0,1,0,-1,0,-1,0,1,0,1,0,-1,0,0,0,0,0], dtype=int)
# P_1 = (2 * P_1).astype(int)

# def monoid_hilbert_basis(CR, P1, P2, mods=(8, 2)):
#     """Hilbert basis of {B >= 0 : CR·B = 0, P1·B ≡ 0 (mod 4), P2·B ≡ 0 (mod 2)}."""
#     CR = np.atleast_2d(np.asarray(CR, dtype=int))
#     n  = CR.shape[1]

#     equations    = CR.tolist()                              # CR·B = 0
#     inequalities = np.eye(n, dtype=int).tolist()            # B_i >= 0  (makes the cone pointed)
#     congruences  = [list(map(int, P1)) + [int(mods[0])],    # P1·B ≡ 0 (mod 4)
#                     list(map(int, P2)) + [int(mods[1])]]    # P2·B ≡ 0 (mod 2)

#     C = Cone(equations=equations,
#              inequalities=inequalities,
#              congruences=congruences)
#     H = np.array(C.HilbertBasis(), dtype=int)               # one generator per row
#     return H

# H = monoid_hilbert_basis(CR, P_1, P_2)
# print("Hilbert basis: ", H.shape[0], "generators, dim", H.shape[1])

## Below is some attempt to avoid using PyNormaliz
However, either does it take a long time or it requires a huge amount of memory. I don't yet see the result of this part...

In [5]:
import numpy as np

# for z_{4S} indicator
P_1 = np.array([0,0,0,0,1.5,-0.5,-1.5,0.5,0,0,0,0,1.5,-0.5,-1.5,0.5,-1.0,1.0,0,0])
P_2 = np.array([-1,0,1,0,1,0,-1,0,-1,0,1,0,1,0,-1,0,0,0,0,0], dtype=int)
P_1 = (2 * P_1).astype(int)        # clear halves; modulus 4 -> 8 accordingly

def _hilbert_basis_kernel(A):
    """Contejean-Devie: minimal nonneg integer solutions of A x = 0, x >= 0, x != 0."""
    A = [[int(v) for v in row] for row in np.asarray(A, dtype=int).tolist()]
    m = len(A)
    n = len(A[0]) if m else 0
    cols = [[A[i][j] for i in range(m)] for j in range(n)]

    def matvec(x):
        return [sum(A[i][j] * x[j] for j in range(n)) for i in range(m)]

    def dot(a, b):
        return sum(a[i] * b[i] for i in range(m))

    solutions = []

    def dominated(y):
        return any(all(y[k] >= s[k] for k in range(n)) for s in solutions)

    frontier = []
    for j in range(n):
        e = [0] * n
        e[j] = 1
        frontier.append(e)

    seen = set()
    while frontier:
        nxt = []
        for x in frontier:
            Ax = matvec(x)
            if not any(Ax):                       # Ax == 0  -> solution
                if not dominated(x):
                    solutions.append(x)
                continue
            for j in range(n):
                if dot(Ax, cols[j]) < 0:          # Contejean-Devie criterion
                    y = list(x)
                    y[j] += 1
                    key = tuple(y)
                    if key in seen or dominated(y):
                        continue
                    seen.add(key)
                    nxt.append(y)
        frontier = nxt
    return np.array(solutions, dtype=int).reshape(-1, n)

def monoid_hilbert_basis(CR, P1, P2, mods=(8, 2)):
    """Hilbert basis of {B >= 0 : CR·B = 0, P1·B ≡ 0 (mod mods[0]), P2·B ≡ 0 (mod mods[1])}.
    Congruences via free slacks u = u+ - u-, v = v+ - v-."""
    CR = np.atleast_2d(np.asarray(CR, dtype=int))
    m, n = CR.shape
    m1, m2 = mods

    # augmented columns: [ B (n) | u+ | u- | v+ | v- ]
    top = np.hstack([CR, np.zeros((m, 4), dtype=int)])
    r1  = np.hstack([np.asarray(P1, dtype=int).reshape(1, -1),
                     np.array([[-m1,  m1, 0, 0]])])
    r2  = np.hstack([np.asarray(P2, dtype=int).reshape(1, -1),
                     np.array([[0, 0, -m2, m2]])])
    A = np.vstack([top, r1, r2])

    Haug = _hilbert_basis_kernel(A)
    H = Haug[:, :n]                       # project onto B
    H = H[np.any(H != 0, axis=1)]         # drop zeros
    H = np.unique(H, axis=0)              # dedup
    return H

H = monoid_hilbert_basis(CR, P_1, P_2)
print("Hilbert basis:", H.shape[0], "generators, dim", H.shape[1])

KeyboardInterrupt: 

In [ ]:
# # Try to speed up
# import numpy as np

# P_1 = np.array([0,0,0,0,1.5,-0.5,-1.5,0.5,0,0,0,0,1.5,-0.5,-1.5,0.5,-1.0,1.0,0,0])
# P_2 = np.array([-1,0,1,0,1,0,-1,0,-1,0,1,0,1,0,-1,0,0,0,0,0], dtype=int)
# P_1 = (2 * P_1).astype(int)        # clear halves; modulus 4 -> 8

# def _hilbert_basis_kernel(A):
#     """Contejean-Devie, vectorized. Minimal nonneg integer solutions of A x = 0, x >= 0, x != 0."""
#     A = np.asarray(A, dtype=np.int64)
#     m, N = A.shape
#     AT = A.T.copy()                    # (N, m): row j is a_j, for incremental Ax updates

#     # frontier nodes and their cached Ax = A @ x
#     X  = np.eye(N, dtype=np.int64)     # unit vectors e_j
#     AX = AT.copy()                     # A e_j = a_j

#     solutions = np.empty((0, N), dtype=np.int64)
#     seen = set()

#     def filter_dominated(Y):
#         """Drop rows of Y that are componentwise >= some existing solution."""
#         if solutions.shape[0] == 0 or Y.shape[0] == 0:
#             return Y
#         # (c, s, N) >= test, reduced over coords then over solutions
#         dom = (Y[:, None, :] >= solutions[None, :, :]).all(axis=2).any(axis=1)
#         return Y[~dom]

#     while X.shape[0]:
#         zero = ~AX.any(axis=1)         # rows with Ax == 0  -> candidate solutions
#         if zero.any():
#             new_sol = filter_dominated(X[zero])
#             if new_sol.shape[0]:
#                 solutions = np.vstack([solutions, new_sol])

#         live = X[~zero]
#         AXl  = AX[~zero]
#         if live.shape[0] == 0:
#             break

#         # Contejean-Devie scores: S[i, j] = <A x_i, a_j> = (AXl @ A)[i, j]
#         S = AXl @ A                    # (k, N)
#         rows, cols = np.where(S < 0)   # branch only where criterion holds
#         if rows.size == 0:
#             break

#         children   = live[rows].copy()
#         children[np.arange(rows.size), cols] += 1
#         AX_children = AXl[rows] + AT[cols]   # incremental update, no rematmul

#         # prune: dominated by an existing solution, then dedup against `seen`
#         keep = np.ones(rows.size, dtype=bool)
#         if solutions.shape[0]:
#             dom = (children[:, None, :] >= solutions[None, :, :]).all(axis=2).any(axis=1)
#             keep &= ~dom
#         children, AX_children = children[keep], AX_children[keep]

#         uniq, idx = [], []
#         for i, c in enumerate(map(tuple, children)):
#             if c not in seen:
#                 seen.add(c)
#                 idx.append(i)
#         if not idx:
#             break
#         idx = np.asarray(idx)
#         X, AX = children[idx], AX_children[idx]

#     return solutions

# def monoid_hilbert_basis(CR, P1, P2, mods=(8, 2)):
#     CR = np.atleast_2d(np.asarray(CR, dtype=np.int64))
#     m, n = CR.shape
#     m1, m2 = mods
#     top = np.hstack([CR, np.zeros((m, 4), dtype=np.int64)])
#     r1  = np.hstack([np.asarray(P1, dtype=np.int64).reshape(1, -1), np.array([[-m1,  m1, 0, 0]])])
#     r2  = np.hstack([np.asarray(P2, dtype=np.int64).reshape(1, -1), np.array([[0, 0, -m2, m2]])])
#     A = np.vstack([top, r1, r2])

#     H = _hilbert_basis_kernel(A)[:, :n]
#     H = H[np.any(H != 0, axis=1)]
#     return np.unique(H, axis=0)

# H = monoid_hilbert_basis(CR, P_1, P_2)
# print("Hilbert basis:", H.shape[0], "generators, dim", H.shape[1])

MemoryError: Unable to allocate 22.6 GiB for an array with shape (56158784, 18, 24) and data type bool

## Verify the result of H

In [4]:
import numpy as np
from itertools import product
from pathlib import Path

# SI rows (same convention as find_basis.py)
P_1 = np.array([0, 0, 0, 0, 1.5, -0.5, -1.5, 0.5, 0, 0, 0, 0, 1.5, -0.5, -1.5, 0.5, -1.0, 1.0, 0, 0])
P_2 = np.array([-1, 0, 1, 0, 1, 0, -1, 0, -1, 0, 1, 0, 1, 0, -1, 0, 0, 0, 0, 0], dtype=int)
P_1 = (2 * P_1).astype(int)

H = np.load(Path("hilbert_basis.npy"))
print("loaded H:", H.shape[0], "generators, dim", H.shape[1])

BLOCKS = [
    ("Gamma", [0, 1, 2, 3]),
    ("A", [4, 5, 6, 7]),
    ("M", [8, 9, 10, 11]),
    ("Z", [12, 13, 14, 15]),
    ("R", [16, 17]),
    ("X", [18, 19]),
]


def in_monoid(b, CR, P1, P2, mods=(8, 2)):
    CR = np.atleast_2d(np.asarray(CR, dtype=int))
    b = np.asarray(b, dtype=int)
    return (
        np.all(b >= 0)
        and np.all(CR @ b == 0)
        and (P1 @ b) % mods[0] == 0
        and (P2 @ b) % mods[1] == 0
    )


def verify_generators(H, CR, P1, P2, mods=(8, 2)):
    ok = all(in_monoid(h, CR, P1, P2, mods) for h in H)
    print("membership in monoid:", ok)
    return ok


def verify_unique(H):
    n_unique = len({tuple(map(int, h)) for h in H})
    ok = n_unique == len(H)
    print("no duplicate rows:", ok, f"({n_unique} unique of {len(H)})")
    return ok


def verify_minimal(H):
    """Each generator is indecomposable among the others (necessary for a Hilbert basis)."""
    bad = []
    for i, h in enumerate(H):
        if decompose(h, np.delete(H, i, axis=0)) is not None:
            bad.append(i)
    ok = len(bad) == 0
    print("generators indecomposable among themselves:", ok, f"({len(bad)} bad)")
    return ok


def decompose(b, H, memo=None):
    """Return coeffs c >= 0 with c @ H = b, or None."""
    b = tuple(int(x) for x in b)
    if all(x == 0 for x in b):
        return np.zeros(len(H), dtype=int)
    if memo is None:
        memo = {}
    if b in memo:
        return memo[b]
    for i, h in enumerate(H):
        h = tuple(int(x) for x in h)
        if all(h[j] <= b[j] for j in range(len(b))):
            sub = decompose(tuple(b[j] - h[j] for j in range(len(b))), H, memo)
            if sub is not None:
                c = sub.copy()
                c[i] += 1
                memo[b] = c
                return c
    memo[b] = None
    return None


def compositions(total, parts):
    if parts == 1:
        yield (total,)
        return
    for first in range(total + 1):
        for rest in compositions(total - first, parts - 1):
            yield (first,) + rest


def enumerate_band_data(N, CR, P1, P2, mods=(8, 2)):
    CR = np.atleast_2d(np.asarray(CR, dtype=int))
    n = CR.shape[1]
    P1 = np.asarray(P1, dtype=int)
    P2 = np.asarray(P2, dtype=int)
    m1, m2 = mods
    block_choices = [list(compositions(N, len(idx))) for _, idx in BLOCKS]
    results = []
    for combo in product(*block_choices):
        B = np.zeros(n, dtype=int)
        for (_, idx), vals in zip(BLOCKS, combo):
            B[idx] = vals
        if np.any(CR @ B != 0):
            continue
        if (P1 @ B) % m1 != 0 or (P2 @ B) % m2 != 0:
            continue
        results.append(B)
    return np.array(results, dtype=int).reshape(-1, n)


def verify_spans_small_band_data(H, CR, P1, P2, mods=(8, 2)):
    """Every 1- and 2-band monoid element lies in N*H (spot-check of completeness)."""
    samples = np.vstack([
        enumerate_band_data(1, CR, P1, P2, mods),
        enumerate_band_data(2, CR, P1, P2, mods),
    ])
    memo = {}
    missing = [b for b in samples if decompose(b, H, memo) is None]
    ok = len(missing) == 0
    print(
        "all 1- and 2-band monoid elements in N*H:",
        ok,
        f"({len(samples)} checked, {len(missing)} missing)",
    )
    return ok


checks = [
    verify_generators(H, CR, P_1, P_2),
    verify_unique(H),
    verify_minimal(H),
    verify_spans_small_band_data(H, CR, P_1, P_2),
]
print("H passes Hilbert-basis checks:", all(checks))

loaded H: 2466 generators, dim 20
membership in monoid: True
no duplicate rows: True (2466 unique of 2466)
generators indecomposable among themselves: True (0 bad)
all 1- and 2-band monoid elements in N*H: True (454 checked, 0 missing)
H passes Hilbert-basis checks: True


## Enumerate $B_1$, $B_2$
We've already got the basis for arbitrary $B$. Next, we'll find all the $B_1,B_2$.
Next, enumerate all the one-band and two-band symmetry data, under constraint $CR\cdot B_i=0, P_1B_i-4u=0,P_2B_i-2v=0$ and $B_1[0]+B_1[1]+B_1[2]+B_1[3]=1,B_2[0]+B_2[1]+B_2[2]+B_2[3]=2$(given compatibility relation, we only need to count the band numbers at $\Gamma$).

In [5]:
import numpy as np
from itertools import product

# Reduced double-valued basis (20 entries), grouped by maximal k point.
BLOCKS = [
    ("Gamma", [0, 1, 2, 3]),
    ("A",     [4, 5, 6, 7]),
    ("M",     [8, 9, 10, 11]),
    ("Z",     [12, 13, 14, 15]),
    ("R",     [16, 17]),
    ("X",     [18, 19]),
]

def compositions(total, parts):
    """Non-negative integer tuples of length `parts` summing to `total`."""
    if parts == 1:
        yield (total,)
        return
    for first in range(total + 1):
        for rest in compositions(total - first, parts - 1):
            yield (first,) + rest

def enumerate_band_data(N, CR, P1, P2, mods=(8, 2)):
    """All B >= 0 with band number N at Gamma (hence every block sums to N),
    CR·B = 0, P1·B ≡ 0 (mod mods[0]), P2·B ≡ 0 (mod mods[1]).

    P1 is the *scaled* integer row (= 2 × the half-integer SI row), so the
    z_4S = 0 condition is P1·B ≡ 0 (mod 8); P2·B ≡ 0 (mod 2) gives delta_2S = 0.
    """
    CR = np.atleast_2d(np.asarray(CR, dtype=int))
    n  = CR.shape[1]
    P1 = np.asarray(P1, dtype=int)
    P2 = np.asarray(P2, dtype=int)
    m1, m2 = mods

    block_choices = [list(compositions(N, len(idx))) for _, idx in BLOCKS]

    results = []
    for combo in product(*block_choices):
        B = np.zeros(n, dtype=int)
        for (_, idx), vals in zip(BLOCKS, combo):
            B[idx] = vals
        if np.any(CR @ B != 0):          # finer compatibility relations
            continue
        if (P1 @ B) % m1 != 0:           # z_4S = 0
            continue
        if (P2 @ B) % m2 != 0:           # delta_2S = 0
            continue
        results.append(B)
    return np.array(results, dtype=int).reshape(-1, n)

B1 = enumerate_band_data(1, CR, P_1, P_2)   # single band: z_2 is automatically 0
B2 = enumerate_band_data(2, CR, P_1, P_2)   # two bands:   z_2 free
print("B1:", B1.shape[0], "single-band vectors")
print("B2:", B2.shape[0], "two-band vectors")

# Generator matrix G (columns = the B1, B2 vectors) for the GU = H + EV step.
G = np.vstack([B1, B2]).T
print("G shape:", G.shape)

B1: 16 single-band vectors
B2: 438 two-band vectors
G shape: (20, 454)


In [9]:
# We could verify that $B_1$ has vanishing z_2 
# (it already has vanishing z_{4S} and \delta_{2S})
for i in range(16):
    print(f"{i}: z2 = ", 0.5 * (B1[i][1]+B1[i][5]+B1[i][9]+B1[i][13] - (B1[i][0]+B1[i][4]+B1[i][8]+B1[i][12])))
# indeed, all mod2=0

0: z2 =  0.0
1: z2 =  0.0
2: z2 =  0.0
3: z2 =  0.0
4: z2 =  0.0
5: z2 =  0.0
6: z2 =  0.0
7: z2 =  0.0
8: z2 =  2.0
9: z2 =  0.0
10: z2 =  0.0
11: z2 =  0.0
12: z2 =  0.0
13: z2 =  0.0
14: z2 =  0.0
15: z2 =  -2.0


## Finally, let's check whether $B_1,B_2$ could be combined to cover $H$
This is done by solving $G\cdot U_i = h_i + EBR \cdot V_i$ for non-negative integer vectors $U_i$ and $V_i$. $H=(h_1,\cdots h_{2466})$.
In fact, 

In [23]:
# GU = h + EV  for every h in H, with U, V >= 0 integer.
# Requires: pip install highspy
# Uses one persistent HiGHS model per worker; only row bounds (RHS) change between solves.
import os
import time

import numpy as np

from gu_cover_solver import solve_all

H = np.load("hilbert_basis.npy").astype(int)        # (2466, 20)
E = np.asarray(EBR, dtype=int)                       # (20, 36)
G = G.astype(int)                                    # (20, 454) from the previous cell

nU, nV = G.shape[1], E.shape[1]
# A x = h with x = [U; V],  A = [G | -E]
A = np.ascontiguousarray(np.hstack([G, -E]), dtype=np.float64)

N_WORKERS = min(os.cpu_count() or 1, 8)
t0 = time.time()

try:
    feasible, U_all, V_all, failures = solve_all(
        H, A, nU,
        n_workers=N_WORKERS,
        parallel="process",          # one persistent HiGHS model per process
        chunk_size=40,
        t0=t0,
    )
except Exception as exc:
    # Jupyter on Windows can block process pools; threads still keep warm HiGHS models.
    print(f"process pool unavailable ({exc!r}); using thread pool")
    feasible, U_all, V_all, failures = solve_all(
        H, A, nU,
        n_workers=N_WORKERS,
        parallel="thread",
        chunk_size=40,
        t0=t0,
    )

# Spot-check a few solutions.
for i in np.where(feasible)[0][:3]:
    assert np.all(G @ U_all[i] - E @ V_all[i] == H[i])

print(f"\nfeasible: {feasible.sum()} / {len(H)}")
print(f"infeasible generators: {failures}")
print(f"claim GU = h + EV holds for all h in H: {len(failures) == 0}")
print(f"elapsed: {time.time() - t0:.1f}s")

  200/2466 done, 200 feasible, 99.0s
  400/2466 done, 400 feasible, 114.9s
  600/2466 done, 600 feasible, 189.2s
  800/2466 done, 800 feasible, 222.8s
  1000/2466 done, 1000 feasible, 306.9s
  1200/2466 done, 1200 feasible, 366.7s
  1400/2466 done, 1400 feasible, 408.7s
  1600/2466 done, 1600 feasible, 480.2s
  1800/2466 done, 1800 feasible, 529.0s
  2000/2466 done, 2000 feasible, 627.5s
  2200/2466 done, 2200 feasible, 692.3s
  2400/2466 done, 2400 feasible, 750.3s
  2466/2466 done, 2466 feasible, 758.7s

feasible: 2466 / 2466
infeasible generators: []
claim GU = h + EV holds for all h in H: True
elapsed: 758.8s
